# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their @id and name
print("Available record sets:")
for record_set in dataset.record_sets:
    print(f"@id: {record_set['@id']}, name: {record_set.get('name', 'N/A')}")

# For each record set, print field @ids and names
for record_set in dataset.record_sets:
    print(f"\nFields in record set @{record_set['@id']}:")
    for field in record_set.get('fields', []):
        print(f"  Field @id: {field['@id']}, name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# ---
# Example: Load all record sets into dataframes by their @id
# ---
record_sets_ids = [record_set['@id'] for record_set in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    try:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set @{record_set_id}")
    except Exception as e:
        print(f"Failed to load record set @{record_set_id}: {e}")

# For demonstration, we'll use the first available record set.
if len(record_sets_ids) > 0:
    main_rs_id = record_sets_ids[0]
    print(f"Columns in main record set @{main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# We'll demonstrate EDA on the main record set.
# To do so, identify a numeric field. We'll try a few likely column names (by their @id in schema):

numeric_field_candidates = ['coverage', 'MW', 'norm_abundance', 'pI', 'peptide_count']
main_df = dataframes[main_rs_id]

# Select the first available numeric field
for col in numeric_field_candidates:
    if col in main_df.columns:
        numeric_field = col
        break
else:
    raise ValueError("No numeric field found in main record set for EDA.")

print(f"Using field '{numeric_field}' for EDA.")

# Filter records where numeric_field > threshold
threshold = main_df[numeric_field].mean()  # Use mean as an example threshold
filtered_df = main_df[main_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a text/categorical field if present
group_field_candidates = ['accession', 'description', 'sample_name']
for group_col in group_field_candidates:
    if group_col in main_df.columns:
        group_field = group_col
        break
else:
    group_field = None

if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the main numeric field
plt.figure(figsize=(8, 4))
sns.histplot(main_df[numeric_field], kde=True)
plt.title(f"Distribution of {numeric_field}" )
plt.xlabel(numeric_field)
plt.show()

# If two numeric fields available, plot a scatter
additional_numeric = [col for col in main_df.columns if col != numeric_field and pd.api.types.is_numeric_dtype(main_df[col])]
if additional_numeric:
    plt.figure(figsize=(6, 6))
    sns.scatterplot(data=main_df, x=numeric_field, y=additional_numeric[0])
    plt.title(f"{numeric_field} vs {additional_numeric[0]}")
    plt.xlabel(numeric_field)
    plt.ylabel(additional_numeric[0])
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset provides mass spectrometry protein analysis for extracellular vesicles derived from stimulated human mast cells.
- We demonstrated loading and basic EDA using the `mlcroissant` library—listing the record set and field `@id`s, extracting data into DataFrames, filtering and normalizing numeric data, and visualizing distributions.
- This notebook can be extended for advanced statistical analyses, machine learning, or domain-specific protein feature discovery using the referenced entity `@id`s for strict provenance.
